# v2.02 — v2.01 + curve-noise fixes (validation only)

All values chosen by Siwat on 2026-10-24/25 (`configs/v2_02.json`; record: `claude/log-entry-2026-10-25-v2-02-decisions.md`). Changes from v2.01:
- patches **2 / 5 / 21** (stride = length), SMA and EMA **9 / 21 / 60**, batch **128**;
- **cosine** learning rate 1e-4 → 0 over **30 epochs, no early stopping**;
- **EMA** of the weights (decay 0.99); BatchNorm statistics recomputed for them; only the EMA weights are scored, kept and saved;
- kept epoch = best mean macro-F1 among epochs **up to the lowest hold-out cross-entropy**;
- per-epoch diagnostics: hold-out cross-entropy, predicted-class shares, calls unchanged from the previous epoch;
- the 3-seed **ensemble** reported alongside the per-seed mean ± std.

**Runtime: A100** (Siwat). An L4 also works; results go to a folder per GPU (`results/v2xx/A100/` or `L4/`). Every seed runs all 30 epochs; each epoch prints the hold-out scores, the calls, and `<== KEPT` when the kept weights change (saved at once to Drive). Test block stays sealed.

In [ ]:
!nvidia-smi -L
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
mem = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(f'GPU: {name}, {mem:.0f} GB')
GPU_TAG = 'A100' if 'A100' in name else 'L4' if 'L4' in name else None
assert GPU_TAG, 'stop: switch the runtime to an A100 or L4 GPU (Runtime -> Change runtime type)'
print('results folder tag:', GPU_TAG)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT = f'{SENIOR}/prices', f'{SENIOR}/results/v2xx/{GPU_TAG}'   # one folder per GPU
RUN_WALK_FORWARD = False   # Siwat, 2026-10-19: official window only
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, json, subprocess, sys
from pathlib import Path
for f in ['scripts/train_v2xx.py', 'src/proposal_model.py', 'src/encoders.py', 'src/features_proposal.py', 'src/train_utils.py',
          'src/stopping.py', 'src/run_config.py', 'configs/v2_02.json', 'tests/test_v2xx.py', 'tests/test_v2xx_training.py',
          'tests/test_v2xx_pipeline.py', 'scripts/plot_v2xx_training.py']:
    assert os.path.exists(f), f'stop: push the v2.02 code first ({f} missing)'
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])

In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# 3. Config must be complete
sys.path.insert(0, 'src')
import run_config as RC
cfg = RC.load('configs/v2_02.json')
print('v2.02 ready:', json.dumps({k: cfg[k] for k in ('model', 'loss', 'training', 'report')}, indent=1))

In [ ]:
# 4. Tests (about 6 minutes); the notebook stops if any fails
for t in ['tests/test_v2xx.py', 'tests/test_v2xx_training.py', 'tests/test_v2xx_pipeline.py', 'tests/test_pipeline.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-3000:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed\n' + r.stderr[-2000:]

In [ ]:
# 5. v2.02, official validation window, 3 seeds x 30 epochs
def done(path, n=3):
    try:
        r = json.load(open(path)); return len(r['runs']) == n and 'summary' in r
    except Exception:
        return False
def run(label, config, out, extra=()):
    if done(out):
        print(f'=== {label}: already done'); return
    print(f'=== {label}'); sys.stdout.flush()
    p = subprocess.Popen(['python', '-u', 'scripts/train_v2xx.py', '--config', config, '--out', out, *extra],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'{label} failed (exit {p.returncode}); the error is printed just above')
V202 = f'{OUT}/v2_02.json'
run('v2.02 official window', 'configs/v2_02.json', V202)

In [ ]:
# 6. Summary beside v2.01 (L4 run, 2026-10-20), v2 and the floors
V201 = {'h1': 0.338, 'h5': 0.366, 'h10': 0.368}; V2 = {'h1': 0.387, 'h5': 0.396, 'h10': 0.378}
BAR = {'h1': 0.393, 'h5': 0.400, 'h10': 0.407}
r = json.load(open(V202)); s, fl = r['summary']['model'], r['floors']; ens = r.get('seed_ensemble', {}).get('val', {})
print(f"{'':6}{'v2.02 F1':>16}{'ensemble':>10}{'v2.01':>7}{'v2':>7}{'bar':>7}{'continues':>11}{'reverses':>10}{'R2_OS':>10}{'RMSE baht':>11}{'persist.':>10}")
for h in ('h1', 'h5', 'h10'):
    m = s[h]
    print(f"{h:6}{m['macro_f1']['mean']:>9.3f} ± {m['macro_f1']['std']:.3f}{ens.get(h, {}).get('macro_f1', float('nan')):>10.3f}"
          f"{V201[h]:>7.3f}{V2[h]:>7.3f}{BAR[h]:>7.3f}{fl[h]['trend_continues']['macro_f1']:>11.3f}{fl[h]['trend_reverses']['macro_f1']:>10.3f}"
          f"{m['r2_os_vs_zero']['mean']:>+10.4f}{m['price_rmse_baht']['mean']:>11.3f}{fl[h]['persistence_price_baht']['rmse']:>10.3f}")
for x in r['runs']:
    sw = [v for k, v in x.items() if k.startswith('scale_weights')]
    print('seed', x['seed'], 'epochs', x['epochs_run'], 'kept', x['best_epoch'], 'lowest CE epoch', x.get('ce_min_epoch'),
          'best F1 epoch overall', x.get('best_f1_epoch_overall'), 'params', x['params']['total'],
          'GPU peak GB', x['history'][-1].get('gpu_peak_gb'), 'scale weights', sw[0]['overall'] if sw else '-')

In [ ]:
# 7. Learning curves (+ CE, call agreement) and the call-share figure, saved beside the results on Drive
from IPython.display import Image, display
sys.path.insert(0, 'scripts')
import importlib, plot_v2xx_training as PT
importlib.reload(PT)
for png in (PT.plot(Path(V202)), PT.plot_calls(Path(V202))):
    if png:
        print('saved', png); display(Image(filename=str(png)))